<a href="https://colab.research.google.com/github/sreenadhyadav883/7006SCN2526MAYJUL-7006SCN_SYA_16073319/blob/main/notebooks/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Run this cell once at the start of a fresh Colab runtime
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
import os
import findspark

# Java path used by Colab after installing OpenJDK 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
findspark.init()
from pyspark.sql import SparkSession

# Stop an old Spark session if the cell is re-run
try:
    spark.stop()
except NameError:
    pass
spark = (
    SparkSession.builder
    .appName("7006SCN_PySpark")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

In [1]:
# Loading the dataset from the public mirror
# Used -q (quiet) so it doesn't spam the googlecolab output with download logs
!wget -q https://datasets-documentation.s3.eu-west-3.amazonaws.com/amazon_reviews/amazon_reviews_2015.snappy.parquet

In [5]:
#Load the Data
df = spark.read.parquet("amazon_reviews_2015.snappy.parquet")

#Partition Count
num_partitions = df.rdd.getNumPartitions()
print(f"Current Number of Partitions: {num_partitions}")

Current Number of Partitions: 70


In [6]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import StringType

#Missing Value Handling : Drop rows where the review text or the star rating is missing
clean_df = df.dropna(subset=["review_body", "star_rating"])

#Cast the binary review_body back into readable English text
clean_df = clean_df.withColumn("review_body", col("review_body").cast("string"))

#Target Encoding: Convert 1-5 star ratings into a binary classification label
final_df = clean_df.withColumn("label",
                               when(col("star_rating") >= 4, 1).otherwise(0))

print("CLEANED DATA & ENCODED LABEL")
final_df.select("star_rating", "label", "review_body").show(5, truncate=50)

CLEANED DATA & ENCODED LABEL
+-----------+-----+--------------------------------------------------+
|star_rating|label|                                       review_body|
+-----------+-----+--------------------------------------------------+
|          5|    1|I have made multiple purchases of this prosciut...|
|          3|    0|I am not sure if it's a product or storage prob...|
|          1|    0|I was  hoping this had a stronger taste than re...|
|          5|    1|                                      Awesome Tea!|
|          4|    1|This tasty spread tastes just like a melted Ree...|
+-----------+-----+--------------------------------------------------+
only showing top 5 rows


In [7]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

#Feature Engineering Configuration
#Step1: Splitting the sentences into an array of individual lowercase words
tokenizer = Tokenizer(inputCol="review_body", outputCol="words")

#Step2:Filtering out non-informative English grammar words like "the", "a", "is"
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

#Step3: Converting word arrays into numerical term frequency vectors
#Limiting to 10,000 features (to prevent out of memory errors on Colab)
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)

#Step4:Applying IDF to scale feature weights (penalizes common words and  boosts rare sentiment words)
idf = IDF(inputCol="rawFeatures", outputCol="features")
#Assemble the PySpark Pipeline
pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])

print("Fitting the pipeline and transforming the data... (This may take a moment on 41M rows)")
# Fit the pipeline model to calculate IDF weights across the dataset
pipeline_model = pipeline.fit(final_df)
prepared_df = pipeline_model.transform(final_df)

#Validate Final Output Schema and Features
print("Pipeline logic verification")
prepared_df.select("label", "features").show(5, truncate=120)

Fitting the pipeline and transforming the data... (This may take a moment on 41M rows)
Pipeline logic verification
+-----+------------------------------------------------------------------------------------------------------------------------+
|label|                                                                                                                features|
+-----+------------------------------------------------------------------------------------------------------------------------+
|    1|(10000,[80,592,2245,2752,3344,3372,3506,6274,6532,8839,9179,9475],[3.355214131355807,5.456622455184766,6.554781955423...|
|    0|(10000,[234,387,447,494,1424,1452,1564,1611,1698,1821,1964,2379,3079,3296,3372,3506,3513,3525,3636,3920,4307,4507,491...|
|    0|(10000,[404,712,930,1195,1805,1872,2069,2578,3372,4890,5177,5402,6740,6875,8122,8130,9385],[5.96186869883732,4.099763...|
|    1|                                                               (10000,[6054,9065],[5.752525677267096,4.1